In [1]:
import pandas as pd

df = pd.read_csv("../data/cleaned/pokemon_cleaned.csv")

In [2]:
df['Is_Special'] = df[['Legendary', 'Mythical', 'Pseudo_Legendary']].any(axis=1)

df['Total_Power'] = df[['HP','Attack','Defense','Special_Attack','Special_Defense','Speed']].sum(axis=1)

In [3]:
y = df['Is_Special']

In [4]:
X = df.drop(columns=[
    'Is_Special',
    'Legendary',
    'Mythical',
    'Pseudo_Legendary',
    'Name',
    'Abilities',
    'ID'
], errors='ignore')

In [5]:
X = pd.get_dummies(X, columns=['Type_1', 'Type_2'], drop_first=True)

In [6]:
X = X.select_dtypes(include=['number'])

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [9]:
y_pred = model.predict(X_test)

In [10]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9777777777777777
              precision    recall  f1-score   support

       False       0.99      0.99      0.99       164
        True       0.88      0.88      0.88        16

    accuracy                           0.98       180
   macro avg       0.93      0.93      0.93       180
weighted avg       0.98      0.98      0.98       180



## Model Evaluation: Logistic Regression

The Logistic Regression model achieved an overall accuracy of approximately 97.7%.

However, due to class imbalance (fewer special Pokémon), accuracy alone is not a sufficient metric.

Key observations:
- The model performs very well on non-special Pokémon (majority class)
- For special Pokémon:
  - Precision: 0.88
  - Recall: 0.88

This indicates that the model is able to correctly identify most special Pokémon, although some are still misclassified.

Overall, the model demonstrates strong performance, but there is room for improvement, particularly in identifying all special Pokémon.

In [11]:
import pandas as pd

feature_importance = pd.Series(model.coef_[0], index=X.columns)
feature_importance.sort_values(ascending=False).head(10)

Total_Power        0.044266
Speed              0.020767
Special_Defense    0.011283
Defense            0.010806
HP                 0.005506
Attack            -0.001230
Special_Attack    -0.002866
dtype: float64

## Insight: Feature Importance (Logistic Regression)

The model coefficients suggest that Total Power has the strongest positive influence on predicting whether a Pokémon is special.

Individual stats such as Attack, Special Attack, and Defense show smaller and sometimes negative coefficients. This is likely due to multicollinearity, as Total Power is derived from these stats and captures their combined effect.

Additionally, the presence of features like ID (which have no real predictive meaning) indicates the importance of careful feature selection.

Overall, the model relies more on combined strength rather than any single stat to make predictions.

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [13]:
y_pred_rf = rf_model.predict(X_test)

In [14]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.9777777777777777
              precision    recall  f1-score   support

       False       0.99      0.99      0.99       164
        True       0.88      0.88      0.88        16

    accuracy                           0.98       180
   macro avg       0.93      0.93      0.93       180
weighted avg       0.98      0.98      0.98       180



In [15]:
import pandas as pd

feature_importance_rf = pd.Series(rf_model.feature_importances_, index=X.columns)
feature_importance_rf.sort_values(ascending=False).head(10)

Total_Power        0.559167
Speed              0.092112
Special_Attack     0.084831
Special_Defense    0.077479
HP                 0.075605
Attack             0.058440
Defense            0.052365
dtype: float64

## Model Comparison: Logistic Regression vs Random Forest

Both Logistic Regression and Random Forest models achieved similar performance:

- Accuracy: ~97.7%
- Precision (Special): 0.88
- Recall (Special): 0.88

This suggests that the underlying patterns in the data are strong but limited, and more complex models do not significantly improve performance.

### Feature Importance (Random Forest)

The most important features identified by the Random Forest model are:

- Total Power (dominant feature)
- Speed
- Special Attack
- Special Defense
- HP

This indicates that overall strength (Total Power) plays the most significant role in determining whether a Pokémon is classified as special.

Secondary attributes such as speed and special stats also contribute, suggesting that special Pokémon tend to be both powerful and well-rounded.

Overall, the model confirms that classification is driven by a combination of features rather than any single stat alone.

In [16]:
rf_model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

In [17]:
y_pred = model.predict(X_test)

In [18]:
y_prob = model.predict_proba(X_test)[:, 1]

In [19]:
y_pred_adjusted = (y_prob > 0.3).astype(int)

## Improving Recall for Special Pokémon

The initial models achieved strong overall accuracy but missed some special Pokémon (recall = 0.88).

To improve recall, two approaches can be applied:

1. **Class Weight Adjustment**  
   By assigning higher importance to the minority class (special Pokémon), the model becomes more sensitive to detecting them.

2. **Threshold Adjustment**  
   Lowering the decision threshold increases the likelihood of classifying a Pokémon as special, improving recall at the cost of precision.

These techniques highlight the trade-off between precision and recall, where improving one may reduce the other.

In [20]:
rf_model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

In [21]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

       False       0.99      0.99      0.99       164
        True       0.88      0.88      0.88        16

    accuracy                           0.98       180
   macro avg       0.93      0.93      0.93       180
weighted avg       0.98      0.98      0.98       180



In [22]:
y_prob[:20]

array([3.08963852e-03, 7.01875638e-01, 2.50877842e-05, 1.99590688e-02,
       5.51073951e-04, 2.63253447e-07, 7.77437206e-01, 3.58241285e-03,
       6.99405088e-07, 9.84406046e-05, 1.42773644e-05, 1.09403210e-01,
       5.55820254e-04, 4.14269463e-06, 2.66613052e-01, 3.94610199e-02,
       5.49930392e-03, 3.13620724e-04, 8.55456194e-01, 7.66486601e-06])

In [23]:
import numpy as np

np.min(y_prob), np.max(y_prob)

(np.float64(7.110312079355901e-09), np.float64(0.9944205515684499))

In [24]:
y_pred_adjusted = (y_prob > 0.1).astype(int)

In [25]:
print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

       False       0.99      0.91      0.95       164
        True       0.48      0.88      0.62        16

    accuracy                           0.91       180
   macro avg       0.73      0.89      0.78       180
weighted avg       0.94      0.91      0.92       180



## Threshold Adjustment Analysis

Lowering the decision threshold from 0.5 to 0.1 did not improve recall for special Pokémon, which remained at 0.88.

However, precision dropped significantly from 0.88 to 0.48, and overall accuracy decreased.

This indicates that the model is already making confident predictions, and the remaining misclassified cases are not borderline instances that can be recovered by adjusting the threshold.

Instead, these errors likely arise from inherent overlap in the feature space, where some non-special Pokémon resemble special ones and vice versa.

This highlights a key limitation of the dataset and feature set.

In [26]:
results = X_test.copy()

results['Actual'] = y_test.values
results['Predicted'] = y_pred_rf   

misclassified = results[results['Actual'] != results['Predicted']]

misclassified.head()

,HP,Attack,Defense,Special_Attack,Special_Defense,Speed,Total_Power,Actual,Predicted
361,80,80,80,80,80,80,480,False,True
807,46,65,65,55,35,34,300,True,False
772,95,95,95,95,95,95,570,False,True
789,43,29,131,29,131,37,400,True,False


In [27]:
original = df.loc[misclassified.index]

original[['Name', 'Type_1', 'Total_Power', 'Is_Special']]

,Name,Type_1,Total_Power,Is_Special
361,glalie,ice,480,False
807,meltan,steel,300,True
772,silvally,normal,570,False
789,cosmoem,psychic,400,True


In [28]:
false_negatives = original[
    (results['Actual'] == True) & (results['Predicted'] == False)
]

false_positives = original[
    (results['Actual'] == False) & (results['Predicted'] == True)
]

C:\Users\Mayank\AppData\Local\Temp\ipykernel_6824\1694072504.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  false_negatives = original[
C:\Users\Mayank\AppData\Local\Temp\ipykernel_6824\1694072504.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  false_positives = original[


In [29]:
misclassified = X_test.copy()

misclassified['Actual'] = y_test.values
misclassified['Predicted'] = y_pred_rf

misclassified_original = df.loc[misclassified.index]

misclassified_original['Actual'] = misclassified['Actual']
misclassified_original['Predicted'] = misclassified['Predicted']

false_negatives = misclassified_original[
    (misclassified_original['Actual'] == True) &
    (misclassified_original['Predicted'] == False)
]

false_positives = misclassified_original[
    (misclassified_original['Actual'] == False) &
    (misclassified_original['Predicted'] == True)
]

In [30]:
false_negatives[['Name', 'Type_1', 'Total_Power']]

,Name,Type_1,Total_Power
807,meltan,steel,300
789,cosmoem,psychic,400


In [31]:
false_positives[['Name', 'Type_1', 'Total_Power']]

,Name,Type_1,Total_Power
361,glalie,ice,480
772,silvally,normal,570


## Insight: Misclassification Analysis

The analysis of misclassified Pokémon reveals two key patterns:

### False Negatives (Missed Special Pokémon)
Some special Pokémon, such as Meltan and Cosmoem, have relatively low total power. As a result, the model fails to identify them as special because they do not follow the general pattern of high strength.

### False Positives (Incorrectly Classified as Special)
Some non-special Pokémon, such as Glalie and Silvally, have high total power. These Pokémon resemble special Pokémon in terms of strength, leading the model to incorrectly classify them.

### Overall Observation
The model relies heavily on strength-related features (Total Power and stats). However, not all special Pokémon are strong, and not all strong Pokémon are special.

This overlap in feature space creates inherent limitations in classification performance, which cannot be fully resolved using the current features.